# Strategy backtest

Uses generated synthetic prices for software research only; these are not actual market data.

Both strategies use the same bars and illustrative transaction costs. Backtests are historical research, not predictions of future performance.


In [ ]:
from pathlib import Path
import sys
repo = Path.cwd()
while not (repo / "backend").is_dir() and repo != repo.parent:
    repo = repo.parent
sys.path.insert(0, str(repo / "backend"))

import pandas as pd
from trading.backtesting.engine import BacktestConfig, BacktestEngine
from trading.backtesting.metrics import calculate_performance_metrics
from trading.data.synthetic import generate_synthetic_ohlcv
from trading.features.feature_pipeline import engineer_features
from trading.strategies.buy_hold import BuyAndHoldStrategy
from trading.strategies.moving_average import MovingAverageCrossoverStrategy
bars = engineer_features(generate_synthetic_ohlcv(periods=1000, frequency="1D", seed=21))
engine = BacktestEngine(BacktestConfig(initial_cash=100_000))
strategies = {"moving_average": MovingAverageCrossoverStrategy(),
              "buy_and_hold": BuyAndHoldStrategy()}
results = {name: engine.run(bars, strategy) for name, strategy in strategies.items()}
metrics = {name: calculate_performance_metrics(result.equity_curve, result.trades,
                                                turnover_value=result.turnover)
           for name, result in results.items()}
pd.DataFrame(metrics).T[["total_return", "cagr", "max_drawdown", "sharpe_ratio",
                         "sortino_ratio", "win_rate", "profit_factor", "trade_count",
                         "turnover", "turnover_ratio"]]
